### **GPU reductions 1**

See the python script.

```python
import sys
import numpy as np
from numba import cuda

##############################
TPB = 64  # Threads per block

@cuda.jit
def reduce_kernel(data, out, n):
    # Get the 1D grid and block indices
    tid = cuda.threadIdx.x
    i = cuda.grid(1)

    # Do reduction for threadblock
    s = 1
    while s < cuda.blockDim.x:
        if tid % (2 * s) == 0 and i + s < n:
            data[i] += data[i + s]
        s *= 2
        cuda.syncthreads()  # Ensure block is synchronized

    # Write result for this block to global memory
    if tid == 0:
        out[cuda.blockIdx.x] = data[i]

def get_grid(n, tpb):
    return (n + (tpb - 1)) // tpb  # Blocks per grid

def reduce(x):
    n = len(x)
    bpg = get_grid(n, TPB)
    out = cuda.device_array(bpg, dtype=x.dtype)
    while bpg > 1:
        reduce_kernel[bpg, TPB](x, out, n)
        n = bpg
        bpg = get_grid(n, TPB)
        x[:n] = out[:n]
    reduce_kernel[bpg, TPB](x, out, n)
    return out
##############################

def number_sum(n):
    random_floats32 = np.random.rand(n, ).astype(np.float32)
    sum = reduce(cuda.to_device(random_floats32))
    return sum.copy_to_host()[0]

n = sys.argv[1]
print(number_sum(int(n)))
```

### **GPU reductions 2**

We upload the script from previous task to the HPC and call:
```bash
nsys profile -o GPU_reductions_profile_data python GPU_reductions.py 4000
```
But this we cannot run interactive jobs or access GPU on the HPC we cannot run this wtf? But we could answer:

"The runtime is dominated by host-device memory transfers, while the reduction kernel contributes relatively little execution time."

### **GPU reductions 3**

Now we want to do:

> copy once with `cuda.to_device`

> keep everything on GPU

>copy once back with `.copy_to_host()`

### **GPU reductions 4**

We added this in the kernel function:
```python
@cuda.jit
def reduce_kernel(data, out, n):
    # Shared memory for this block
    sdata = cuda.shared.array(shape=TPB, dtype=cuda.float32)

    # Get the 1D grid and block indices
    tid = cuda.threadIdx.x
    i = cuda.grid(1)

    # Each thread loads one element
    sdata[tid] = data[i] if i < n else 0.0
    cuda.syncthreads() # Ensure all are done

    # Do reduction for threadblock
    s = 1
    while s < cuda.blockDim.x:
        if tid % (2 * s) == 0 and tid + s < cuda.blockDim.x:
            sdata[tid] += sdata[tid + s]
        s *= 2
        cuda.syncthreads()  # Ensure block is synchronized

    # Write result for this block to global memory
    if tid == 0:
        out[cuda.blockIdx.x] = sdata[0]
```

### **GPU reductions 5**

We make the change inside the while loop:
```python
    # Do reduction for threadblock
    s = 1
    while s < cuda.blockDim.x:
        # if tid % (2 * s) == 0 and tid + s < cuda.blockDim.x:
        #   sdata[tid] += sdata[tid + s]

        index = 2 * s * tid
        if (index < cuda.blockDim.x):
            sdata[index] += sdata[index + s]

        s *= 2
        cuda.syncthreads()  # Ensure block is synchronized
```

### **GPU reductions 6**

We have to run this which we are not able to :( given below:
```bash
nsys profile -o profile python GPU_reductions.py 4000
nsys stats profile.nsys-rep
```
Profiling here means to
measure execution, to
break time into components and to
analyze where time is spent.

### **GPU reductions 7**

OPTIONAL.

### **CuPy 1**

We just import cupy as cp and replace numpy with cupy. 

### **CuPy 2**

### **CuPy 3**

Cannot use nsys.

### **CuPy 4**

### **CuPy 5**

Cannot use nsys.